# Iterated frozen covariance

The covariance $M$ is held fixed during a fit (`--frozen-cov`) at the spline-only prediction of a reference point, and the fit is repeated with the previous best fit as reference (`--cov-reference-file`) until no parameter moves by more than `TOL` (prior $\sigma$) between two iterations. Fits: `xml/run_iterated.sh`, outputs in `nuisance_ladder_study_fit_results/<family>_iterfrozen/<variant>/<fit>/`.

Three treatments are compared: **dynamic** ($M$ rebuilt at every tested point, `<family>`), **frozen at CV** (iteration 0; the 2026-11-18 `<family>_fixedcov` trees are the same method from the superseded patch) and **iterated** (the fixed point).

Sections 1–4 need only the iterations (`FINAL_STAGES=""`); section 5 needs the final `profile` stage.

In [ ]:
from dataclasses import replace
from pathlib import Path
import os
import re
import sys

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

start = Path.cwd().resolve()
REPO = next(
    (parent / 'ma_zexp' for parent in (start, *start.parents)
     if (parent / 'ma_zexp' / 'python' / 'scripts' / 'postfit_physical_parameters.py').is_file()),
    None,
)
if REPO is None:
    raise FileNotFoundError('Could not locate ma_zexp/python/scripts/postfit_physical_parameters.py')
if str(REPO / 'python' / 'scripts') not in sys.path:
    sys.path.insert(0, str(REPO / 'python' / 'scripts'))

from postfit_physical_parameters import NUWRO_INPUT_MA, PUBLICATION_RC, SPECS, _fa_curves, load_fit
import make_nuisance_ladder_study_xmls as ladder

mpl.rcParams.update(PUBLICATION_RC)
SPEC = {spec.key: spec for spec in SPECS}

## Controls

In [ ]:
# Method comparison: reads the Neyman / dynamic tree on purpose (the *_fixedcov, *_iterfrozen and
# pearson_/cnp_ outputs only exist there), not the CNP + iterated-frozen analysis root.
OUTPUT_ROOT = Path(os.environ.get('AXIAL_MASS_OUTPUT_ROOT', '/nevis/hopper/data/epelaez/axial_mass'))
STUDY_ROOT = Path(os.environ.get('NUISANCE_LADDER_STUDY_ROOT', OUTPUT_ROOT / 'nuisance_ladder_study_fit_results'))
FAMILIES = ('asimov', 'opendata', 'nuwro')
FITS = ('ma', 'minerva_k6_nuisance')
VARIANTS = ('n5', 'n9', 'n10')            # n9, n10: ladder closure (NuWro only)
STARTS = {'cv': '', 'dynamic': 'dyn_'}   # start reference -> output label of run_iterated.sh
TOL = 0.01                               # convergence threshold, prior sigma
Q2_REFERENCE = 0.50
Q2 = np.linspace(0.0, 2.0, 201)
IREF = int(np.argmin(np.abs(Q2 - Q2_REFERENCE)))
MAX_SAMPLES = 100_000
FIG_DIR = REPO / 'figs' / 'iterated_frozen_covariance'
TABLE_DIR = REPO / 'tables' / 'iterated_frozen_covariance'
FIG_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)

FAMILY_LABEL = {'asimov': 'Asimov', 'opendata': 'Open data', 'nuwro': 'NuWro fake data'}
START_STYLE = {'cv': dict(color='#0072B2', marker='o', ls='-'), 'dynamic': dict(color='#D55E00', marker='s', ls='--')}
NUMBER = r'[-+]?(?:\d+(?:\.\d*)?|\.\d+)(?:[eE][-+]?\d+)?'
CHI2_PATTERN = re.compile(rf'Global Best Fit chi\^2:\s*({NUMBER})')


def fit_dir(family, variant, fit, tree=''):
    """tree: '' (dynamic), '_fixedcov' (frozen at CV, old patch) or '_iterfrozen'."""
    return STUDY_ROOT / f'{family}{tree}' / variant / fit


def read_global_fit(path):
    """<tag>_<out>_global_fit.txt -> Series of best-fit values (prior sigma), in file order."""
    values = {}
    for line in Path(path).read_text().splitlines():
        if ' : ' in line:
            name, value = line.rsplit(' : ', 1)
            values[name.strip()] = float(value)
    return pd.Series(values, dtype=float)


def read_chi2(log):
    found = CHI2_PATTERN.findall(Path(log).read_text(errors='replace')) if Path(log).is_file() else []
    return float(found[-1]) if found else np.nan


print(f'study at {STUDY_ROOT}')

## 1. Trajectories

One row per finished iteration: the best fit of iteration $k$ (parameters in prior $\sigma$), its $\chi^2$ (with $M$ frozen at the reference of that iteration, so not comparable between iterations) and the largest parameter change with respect to the reference it was frozen at. For the `dynamic` start the reference of iteration 0 is the dynamic best fit; for the `cv` start iteration 0 has no previous point.

In [ ]:
rows = []
for family in FAMILIES:
    for variant in VARIANTS:
        for fit in FITS:
            directory = fit_dir(family, variant, fit, '_iterfrozen')
            for start, label in STARTS.items():
                previous = None
                if start == 'dynamic':
                    reference = fit_dir(family, variant, fit) / f'{fit}_v1_global_fit.txt'
                    previous = read_global_fit(reference) if reference.is_file() else None
                k = 0
                while (path := directory / f'{fit}_{label}it{k}_global_fit.txt').is_file():
                    best = read_global_fit(path)
                    delta = (best - previous).abs() if previous is not None and list(previous.index) == list(best.index) else None
                    rows.append(dict(family=family, variant=variant, fit=fit, start=start, iteration=k,
                                     chi2=read_chi2(directory / f'{label}it{k}.log'),
                                     max_delta=delta.max() if delta is not None else np.nan,
                                     moved_most=delta.idxmax() if delta is not None else '', **best))
                    previous, k = best, k + 1
if not rows:
    raise RuntimeError(f'no iteration found below {STUDY_ROOT}/<family>_iterfrozen; run xml/run_iterated.sh')
TRAJ = pd.DataFrame(rows)
KEYS = ['family', 'variant', 'fit', 'start']
PARAMETERS = [c for c in TRAJ.columns if c not in KEYS + ['iteration', 'chi2', 'max_delta', 'moved_most']]
TRAJ.to_csv(TABLE_DIR / 'trajectories.csv', index=False, float_format='%.6g')

last = TRAJ.sort_values('iteration').groupby(KEYS, sort=False).tail(1).set_index(KEYS)
CONVERGENCE = pd.DataFrame({'iterations': last['iteration'], 'last max |delta|': last['max_delta'],
                            'moved most': last['moved_most'], 'converged': last['max_delta'] < TOL}).sort_index()
display(CONVERGENCE.style.format({'last max |delta|': '{:.4f}'}))
display(TRAJ.set_index(KEYS + ['iteration']).sort_index().drop(columns='moved_most').dropna(axis=1, how='all').style.format('{:.4f}', na_rep=''))

In [ ]:
# max |delta| per iteration (log scale) and every parameter against the iteration.
groups = [(family, variant, fit) for family in FAMILIES for variant in VARIANTS for fit in FITS
          if ((TRAJ.family == family) & (TRAJ.variant == variant) & (TRAJ.fit == fit)).any()]
fig, axes = plt.subplots(2, len(groups), figsize=(3.4 * len(groups), 5.6), sharex=True, squeeze=False)
for col, (family, variant, fit) in enumerate(groups):
    sub = TRAJ[(TRAJ.family == family) & (TRAJ.variant == variant) & (TRAJ.fit == fit)]
    names = [p for p in PARAMETERS if sub[p].notna().any()]
    colors = {p: mpl.colormaps['tab10'](i % 10) for i, p in enumerate(names)}
    for start, style in START_STYLE.items():
        run = sub[sub.start == start].sort_values('iteration')
        if run.empty:
            continue
        axes[0][col].plot(run.iteration, run.max_delta, label=f'start: {start}', **style)
        for p in names:
            axes[1][col].plot(run.iteration, run[p], color=colors[p], ls=style['ls'], marker=style['marker'], ms=3, lw=1,
                              label=p if start == sub.start.iloc[0] else None)
    axes[0][col].axhline(TOL, color='grey', lw=0.8, ls=':')
    axes[0][col].set_yscale('log')
    axes[0][col].set_title(f'{FAMILY_LABEL[family]}, {variant}\n{fit}', fontsize=8)
    axes[1][col].set_xlabel('iteration')
    axes[1][col].legend(fontsize=5, ncol=2, frameon=False)
axes[0][0].set_ylabel(r'max $|\Delta\theta|$ to the reference [prior $\sigma$]')
axes[1][0].set_ylabel(r'best fit [prior $\sigma$]')
axes[0][0].legend(fontsize=6, frameon=False)
fig.tight_layout(); fig.savefig(FIG_DIR / 'trajectories.pdf'); plt.show()

## 2. T0: `--frozen-cov` at the CV against the superseded `--fixed-cov-cv` patch

Iteration 0 of the `cv` start is the fit with $M$ frozen at the CV. It should reproduce the best fit and $\chi^2$ of the `<family>_fixedcov` fits of 2026-11-18 to minimizer precision.

In [ ]:
rows = []
for family in FAMILIES:
    for variant in VARIANTS:
        for fit in FITS:
            new = fit_dir(family, variant, fit, '_iterfrozen') / f'{fit}_it0_global_fit.txt'
            old = fit_dir(family, variant, fit, '_fixedcov') / f'{fit}_v1_global_fit.txt'
            if not (new.is_file() and old.is_file()):
                continue
            a, b = read_global_fit(new), read_global_fit(old)
            delta = (a - b.reindex(a.index)).abs()
            chi2_new, chi2_old = read_chi2(new.parent / 'it0.log'), read_chi2(old.parent / 'profile.log')
            rows.append(dict(family=family, variant=variant, fit=fit, max_delta=delta.max(), parameter=delta.idxmax(),
                             chi2_frozen_cov=chi2_new, chi2_fixed_cov_cv=chi2_old, delta_chi2=chi2_new - chi2_old))
T0 = pd.DataFrame(rows)
if T0.empty:
    print('no pair of (iteration 0, old fixed-cov fit) yet')
else:
    display(T0.set_index(['family', 'variant', 'fit']).style.format({c: '{:.4f}' for c in T0.columns if T0[c].dtype.kind == 'f'}))
    T0.to_csv(TABLE_DIR / 't0_frozen_cov_vs_fixed_cov_cv.csv', index=False, float_format='%.6g')

## 3. T2 / T3: the fixed point

Per family and fit: the fixed point reached from the `cv` start, its distance from the fixed point reached from the dynamic best fit (T3: should be below `TOL`-ish), and from the two non-iterated treatments. On Asimov (T2) every column should be zero: the best fit is the truth, so the reference never moves.

In [ ]:
def final_point(family, variant, fit, start):
    run = TRAJ[(TRAJ.family == family) & (TRAJ.variant == variant) & (TRAJ.fit == fit) & (TRAJ.start == start)]
    if run.empty:
        return None
    return run.sort_values('iteration').iloc[-1][PARAMETERS].dropna().astype(float)


def distance(a, b):
    if a is None or b is None:
        return np.nan, ''
    delta = (a - b.reindex(a.index)).abs()
    return delta.max(), delta.idxmax()


rows, points = [], []
for family, variant, fit in groups:
    fixed = final_point(family, variant, fit, 'cv')
    other = final_point(family, variant, fit, 'dynamic')
    dynamic_file = fit_dir(family, variant, fit) / f'{fit}_v1_global_fit.txt'
    dynamic = read_global_fit(dynamic_file) if dynamic_file.is_file() else None
    cv_frozen = TRAJ[(TRAJ.family == family) & (TRAJ.variant == variant) & (TRAJ.fit == fit) & (TRAJ.start == 'cv') & (TRAJ.iteration == 0)]
    cv_frozen = cv_frozen.iloc[0][PARAMETERS].dropna().astype(float) if len(cv_frozen) else None
    base = fixed if fixed is not None else other
    row = dict(family=family, variant=variant, fit=fit)
    for name, point in (('two starts', other if fixed is not None else None), ('vs dynamic', dynamic), ('vs frozen at CV', cv_frozen)):
        row[f'{name}: max |delta|'], row[f'{name}: parameter'] = distance(base, point)
    rows.append(row)
    for name, point in (('dynamic', dynamic), ('frozen at CV', cv_frozen), ('iterated (cv start)', fixed), ('iterated (dynamic start)', other)):
        if point is not None:
            points.append(dict(family=family, variant=variant, fit=fit, treatment=name, **point))
FIXED = pd.DataFrame(rows).set_index(['family', 'variant', 'fit'])
POINTS = pd.DataFrame(points).set_index(['family', 'variant', 'fit', 'treatment'])
display(FIXED.style.format({c: '{:.4f}' for c in FIXED.columns if FIXED[c].dtype.kind == 'f'}, na_rep=''))
display(POINTS.dropna(axis=1, how='all').style.format('{:.3f}', na_rep=''))
FIXED.to_csv(TABLE_DIR / 'fixed_point_distances.csv', float_format='%.6g')
POINTS.to_csv(TABLE_DIR / 'best_fits_by_treatment.csv', float_format='%.6g')

## 4. $F_A$ and the fitted parameters at the fixed point (T4, T5)

Needs the final `profile` stage of `run_iterated.sh` (chain with $M$ frozen at the converged reference). Per treatment: $F_A(Q^2_{\rm ref})$ at the profile best fit, its posterior width, the shift relative to the dynamic fit in units of the dynamic $\sigma[F_A]$ and, for NuWro, the bias against the generator input (dipole, $M_A = 1.03$ GeV).

In [ ]:
TREES = {'dynamic': '', 'frozen at CV': '_fixedcov', 'iterated': '_iterfrozen'}
NUWRO_TRUTH = -1.2723 / (1 + Q2 / NUWRO_INPUT_MA ** 2) ** 2


def variant_spec(fit, variant):
    branches = tuple(branch for branch, _ in ladder.VARIANTS[variant])
    labels = tuple(branch.removesuffix('_UBGenie') for branch in branches)
    spec = SPEC[fit]
    if spec.prior is None:
        return replace(spec, profile_labels=spec.profile_labels + labels, chain_branches=spec.chain_branches + branches)
    return replace(spec, nuisance_labels=spec.nuisance_labels + labels, nuisance_branches=spec.nuisance_branches + branches)


def fa_band(result):
    curves = _fa_curves(result, Q2, max_samples=MAX_SAMPLES)
    best = _fa_curves({'spec': result['spec'], 'samples': np.atleast_2d(result['profile'])}, Q2)[0]
    return dict(best=best, median=np.median(curves, axis=0), sigma=curves.std(axis=0), mean=curves.mean(axis=0))


RESULTS, BANDS = {}, {}
for family, variant, fit in groups:      # only the fits that have iterations
    for treatment, tree in TREES.items():
        try:
            result = load_fit(variant_spec(fit, variant), suite=variant, data_root=STUDY_ROOT / f'{family}{tree}')
        except Exception as error:
            print(f'{family:9s} {variant:4s} {fit:20s} {treatment:13s} skipped: {type(error).__name__}: {error}')
            continue
        if result is None:
            continue
        RESULTS[family, variant, fit, treatment] = result
        BANDS[family, variant, fit, treatment] = fa_band(result)
        flag = '' if result['ess_min'] >= 2000 else '   <-- ESS below 2000'
        print(f'{family:9s} {variant:4s} {fit:20s} {treatment:13s} {len(result["samples"]):>8,} samples, min ESS {result["ess_min"]:>8,.0f}{flag}')

rows, param_rows = [], []
for (family, variant, fit, treatment), band in BANDS.items():
    ref = BANDS.get((family, variant, fit, 'dynamic'))
    row = dict(family=family, variant=variant, fit=fit, treatment=treatment, F_A_best=band['best'][IREF], sigma_F_A=band['sigma'][IREF],
               **{'delta_F [%]': 100 * band['sigma'][IREF] / abs(band['mean'][IREF])})
    if ref is not None:
        row['shift vs dynamic [sigma]'] = (band['best'][IREF] - ref['best'][IREF]) / ref['sigma'][IREF]
        row['sigma ratio'] = band['sigma'][IREF] / ref['sigma'][IREF]
    if family == 'nuwro':
        row['bias vs input [sigma]'] = (band['best'][IREF] - NUWRO_TRUTH[IREF]) / band['sigma'][IREF]
    rows.append(row)
    result = RESULTS[family, variant, fit, treatment]
    for name, entry in result['summary'].iloc[result['joint_indices']].iterrows():
        param_rows.append(dict(family=family, variant=variant, fit=fit, treatment=treatment, parameter=name,
                               best_fit=entry['profile_best_fit'], median=entry['posterior_median'],
                               minus=entry['posterior_median'] - entry['q16'], plus=entry['q84'] - entry['posterior_median']))
if not rows:
    print('no PROfile.root yet')
else:
    index = ['family', 'variant', 'fit', 'treatment']
    FA = pd.DataFrame(rows).set_index(index)
    PARAMS = pd.DataFrame(param_rows).set_index(index + ['parameter'])
    display(FA.style.format('{:.4f}', na_rep=''))
    display(PARAMS.style.format('{:.4f}', na_rep=''))
    FA.to_csv(TABLE_DIR / 'fa_by_treatment.csv', float_format='%.6g')
    PARAMS.to_csv(TABLE_DIR / 'parameters_by_treatment.csv', float_format='%.6g')

In [ ]:
if rows:
    done = sorted({key[:3] for key in BANDS}, key=lambda key: (FAMILIES.index(key[0]), key[1], FITS.index(key[2])))
    colors = {'dynamic': 'black', 'frozen at CV': '#0072B2', 'iterated': '#D55E00'}
    fig, axes = plt.subplots(1, len(done), figsize=(3.4 * len(done), 3.0), sharey=True, squeeze=False)
    for ax, (family, variant, fit) in zip(axes[0], done):
        ref = BANDS.get((family, variant, fit, 'dynamic')) or next(BANDS[key] for key in BANDS if key[:3] == (family, variant, fit))
        with np.errstate(divide='ignore', invalid='ignore'):
            for treatment in TREES:
                band = BANDS.get((family, variant, fit, treatment))
                if band is None:
                    continue
                ax.plot(Q2, band['best'] / ref['best'], color=colors[treatment], lw=1.3, label=f'{treatment}, best fit')
                ax.fill_between(Q2, (band['median'] - band['sigma']) / ref['best'], (band['median'] + band['sigma']) / ref['best'],
                                color=colors[treatment], alpha=0.12, lw=0)
            if family == 'nuwro':
                ax.plot(Q2, NUWRO_TRUTH / ref['best'], color='grey', ls='--', lw=1, label='NuWro input')
        ax.axvline(Q2_REFERENCE, color='grey', lw=0.8, ls=':')
        ax.set_title(f'{FAMILY_LABEL[family]}, {variant}\n{fit}', fontsize=8)
        ax.set_xlabel(r'$Q^2$ [GeV$^2$]')
        ax.set_xlim(0, 1.2); ax.set_ylim(0.8, 1.2)
    axes[0][0].set_ylabel(r'$F_A$ / dynamic best fit')
    axes[0][-1].legend(fontsize=6, frameon=False)
    fig.tight_layout(); fig.savefig(FIG_DIR / 'fa_by_treatment.pdf'); plt.show()

## 5. T6: $\chi^2$ along the nuisance ladder

The best-fit $\chi^2$ per treatment and ladder variant, and the step from the previous finished variant. With the dynamic covariance, floating `NormCCMEC` (n9 $\to$ n10) *raised* $\chi^2$ by 11.6 on NuWro; with $M$ frozen at the CV the step was $+0.8$. In the iterated method the reference prediction contains the shift of a floated knob but not the hidden pull of the same knob when it is treated as a covariance, so the two variants are frozen at different matrices.

In [ ]:
rows = []
for family in FAMILIES:
    for fit in FITS:
        for variant in VARIANTS:
            for treatment, tree in TREES.items():
                chi2 = read_chi2(fit_dir(family, variant, fit, tree) / 'profile.log')
                if np.isfinite(chi2):
                    rows.append(dict(family=family, fit=fit, treatment=treatment, variant=variant, chi2=chi2))
LADDER = pd.DataFrame(rows).pivot(index=['family', 'fit', 'treatment'], columns='variant', values='chi2').reindex(columns=list(VARIANTS))
LADDER = LADDER[LADDER.notna().sum(axis=1) > 1]
for previous, current in zip(VARIANTS[:-1], VARIANTS[1:]):
    LADDER[f'{previous} -> {current}'] = LADDER[current] - LADDER[previous]
display(LADDER.style.format('{:.2f}', na_rep=''))
LADDER.to_csv(TABLE_DIR / 'ladder_chi2_by_treatment.csv', float_format='%.6g')

## 6. Where the two frozen fits land

Frozen at the CV against the iterated fixed point, `n5`, with the dynamic fit in grey for scale. Every floated parameter in prior $\sigma$: marker = profile best fit, bar = 68 % equal-tailed interval of the posterior chain.

In [ ]:
import uproot

LANDING = {'dynamic': dict(color='0.55', marker='D', offset=0.26, label='dynamic'),
           'frozen at CV': dict(color='#0072B2', marker='o', offset=0.0, label='frozen at CV'),
           'iterated': dict(color='#D55E00', marker='s', offset=-0.26, label='iterated (fixed point)')}
PARAMETER_LABEL = {'MACCQE': r'$M_A$', 'FAzexpMinervaK6PCA1': 'PCA 1', 'FAzexpMinervaK6PCA2': 'PCA 2', 'RPA_CCQE': 'RPA CCQE',
                   'FrCEx_N': 'FrCEx N', 'MFP_N': 'MFP N', 'XSecShape_CCMEC': 'MEC shape', 'DecayAngMEC': 'MEC decay angle'}
ORDER = list(PARAMETER_LABEL)
normalise = lambda name: name.removeprefix('weight_spline_').removesuffix('_UBGenie').lower()


def pulls(family, variant, fit, tree, prefix=''):
    """label -> (best fit, q16, q84) in prior sigma, from <fit>_<prefix>v1_PROfile.root."""
    path = fit_dir(family, variant, fit, tree) / f'{fit}_{prefix}v1_PROfile.root'
    if not path.is_file():
        return None
    with uproot.open(path) as f:
        result = f['global_fit_result']
        best = dict(zip(result.axis().labels(), result.values()))
        tree_ = f[next(k for k in f.keys() if k.split(';')[0].endswith('_mcmc_chain'))]
        chain = {normalise(b): tree_[b].array(library='np') for b in tree_.keys()}
    return {label: (value, *np.percentile(chain[normalise(label)], [16, 84])) for label, value in best.items()
            if normalise(label) in chain}


panels = [(family, fit) for fit in FITS for family in ('opendata', 'nuwro')]
fig, axes = plt.subplots(2, 2, figsize=(7.0, 5.6), sharex=True)
for ax, (family, fit) in zip(axes.ravel(), panels):
    sets = {t: pulls(family, 'n5', fit, tree) for t, tree in TREES.items()}
    names = [n for n in ORDER if any(s and n in s for s in sets.values())]
    y_of = {n: len(names) - 1 - i for i, n in enumerate(names)}
    for treatment, style in LANDING.items():
        values = sets[treatment]
        if values is None:
            continue
        for n in names:
            best, q16, q84 = values[n]
            y = y_of[n] + style['offset']
            ax.plot([q16, q84], [y, y], color=style['color'], lw=1.6, solid_capstyle='round')
            ax.plot(best, y, style['marker'], color=style['color'], ms=5.5, mec='white', mew=0.8,
                    label=style['label'] if n == names[0] else None)
    ax.axvline(0, color='0.75', lw=0.8, zorder=0)
    ax.set_yticks([y_of[n] for n in names])
    ax.set_yticklabels([PARAMETER_LABEL[n] for n in names], fontsize=8)
    ax.set_ylim(-0.6, len(names) - 0.4)
    ax.set_title(f'{FAMILY_LABEL[family]}: ' + (r'$M_A$ fit' if fit == 'ma' else r'$z$ expansion, $\bar{\nu}$–H prior'), fontsize=9)
    ax.tick_params(axis='x', labelsize=8)
    ax.grid(axis='x', color='0.92', lw=0.6, zorder=0)
    ax.set_axisbelow(True)
fig.supxlabel(r'best fit and 68 % interval [prior $\sigma$]', fontsize=9)
handles, labels = axes[0][0].get_legend_handles_labels()
fig.legend(handles, labels, loc='upper center', ncol=3, frameon=False, fontsize=8, bbox_to_anchor=(0.5, 1.0))
fig.tight_layout(rect=(0, 0.01, 1, 0.955))
fig.savefig(FIG_DIR / 'landing.pdf'); fig.savefig(FIG_DIR / 'landing.png', dpi=200); plt.show()

## 7. Choice of the statistic: Neyman, Pearson, CNP

The iterated fixed point of the same `n5` fits with the three statistics (`CHI2=pearson` / `CHI2=CNP` in `xml/run_iterated.sh`; outputs prefixed `pearson_` / `cnp_` in the same directories). With `--frozen-cov` the statistical part of $M$ is frozen at the reference prediction as well: Neyman uses the data, Pearson the prediction at the fixed point, CNP the combination $3/(1/d + 2/\mu)$. In PROfit $\mu$ is the prediction with every spline at zero, i.e. the nominal CV for these fits, which have no physics parameters; the fourth series uses the PROfit branch `cnp-shifted-mu` (`--cnp-shifted-mu`), where $\mu$ is the full prediction at the covariance reference, as in Pearson. Same reading as section 6: marker = profile best fit, bar = 68 % interval of the chain, prior $\sigma$.

In [ ]:
METRICS = {'Neyman': dict(prefix='', color='#0072B2', marker='o', offset=0.26),
           'Pearson': dict(prefix='pearson_', color='#D55E00', marker='s', offset=0.0),
           'CNP': dict(prefix='cnp_', color='#009E73', marker='^', offset=-0.26),
           r'CNP, $\mu$ = fitted prediction': dict(prefix='cnpfull_', color='#CC79A7', marker='D', offset=-0.26)}
# Offsets spread over the metrics that have at least one finished fit.
_present = [m for m, style in METRICS.items()
            if any((fit_dir(family, 'n5', fit, '_iterfrozen') / f"{fit}_{style['prefix']}v1_PROfile.root").is_file() for family, fit in panels)]
for i, m in enumerate(_present):
    METRICS[m]['offset'] = 0.3 - 0.6 * i / max(len(_present) - 1, 1)

metric_rows = []
fig, axes = plt.subplots(2, 2, figsize=(7.0, 6.4), sharex=True)
for ax, (family, fit) in zip(axes.ravel(), panels):
    sets = {m: pulls(family, 'n5', fit, '_iterfrozen', style['prefix']) for m, style in METRICS.items()}
    names = [n for n in ORDER if any(s and n in s for s in sets.values())]
    y_of = {n: len(names) - 1 - i for i, n in enumerate(names)}
    for metric, style in METRICS.items():
        values = sets[metric]
        if values is None:
            print(f'{family:9s} {fit:20s} {metric:8s} not there yet')
            continue
        directory = fit_dir(family, 'n5', fit, '_iterfrozen')
        iterations = len(list(directory.glob(f"{fit}_{style['prefix']}it*_global_fit.txt")))
        metric_rows.append(dict(family=family, fit=fit, metric=metric, iterations=iterations,
                                chi2=read_chi2(directory / f"{style['prefix']}profile.log"),
                                **{n: values[n][0] for n in names}))
        for n in names:
            best, q16, q84 = values[n]
            y = y_of[n] + style['offset']
            ax.plot([q16, q84], [y, y], color=style['color'], lw=1.6, solid_capstyle='round')
            ax.plot(best, y, style['marker'], color=style['color'], ms=5.5, mec='white', mew=0.8,
                    label=metric if n == names[0] else None)
    ax.axvline(0, color='0.75', lw=0.8, zorder=0)
    ax.set_yticks([y_of[n] for n in names])
    ax.set_yticklabels([PARAMETER_LABEL[n] for n in names], fontsize=8)
    ax.set_ylim(-0.6, len(names) - 0.4)
    ax.set_title(f'{FAMILY_LABEL[family]}: ' + (r'$M_A$ fit' if fit == 'ma' else r'$z$ expansion, $\bar{\nu}$–H prior'), fontsize=9)
    ax.tick_params(axis='x', labelsize=8)
    ax.grid(axis='x', color='0.92', lw=0.6, zorder=0)
    ax.set_axisbelow(True)
fig.supxlabel(r'best fit and 68 % interval [prior $\sigma$], iterated frozen covariance', fontsize=9)
handles, labels = max((ax.get_legend_handles_labels() for ax in axes.ravel()), key=lambda hl: len(hl[0]))
fig.legend(handles, labels, loc='upper center', ncol=4, frameon=False, fontsize=8, bbox_to_anchor=(0.5, 1.0))
fig.tight_layout(rect=(0, 0.01, 1, 0.955))
fig.savefig(FIG_DIR / 'metric_landing.pdf'); fig.savefig(FIG_DIR / 'metric_landing.png', dpi=200); plt.show()

METRIC_TABLE = pd.DataFrame(metric_rows).set_index(['family', 'fit', 'metric'])
display(METRIC_TABLE.style.format({c: '{:.3f}' for c in METRIC_TABLE.columns if c != 'iterations'}, na_rep=''))
METRIC_TABLE.to_csv(TABLE_DIR / 'best_fits_by_metric.csv', float_format='%.6g')

### $\chi^2$ at the fixed point by statistic

Best-fit $\chi^2$ of the same fits (profile stage, $M$ frozen at the converged reference), one row per fit. These are different statistics evaluated on the same data and model, not nested hypotheses, so the differences are not a $\Delta\chi^2$ test.

In [ ]:
rows_of = {'opendata': FITS, 'nuwro': FITS}
fit_label = {'ma': r'$M_A$ fit', 'minerva_k6_nuisance': '$z$ expansion,\n' + r'$\bar{\nu}$–H prior'}
value = lambda family, fit, metric: (METRIC_TABLE.loc[(family, fit, metric), 'chi2']
                                     if (family, fit, metric) in METRIC_TABLE.index else np.nan)

chi2_rows = [dict(family=family, fit=fit, metric=metric, chi2=value(family, fit, metric),
                  delta_vs_neyman=value(family, fit, metric) - value(family, fit, 'Neyman'))
             for family in rows_of for fit in rows_of[family] for metric in METRICS if np.isfinite(value(family, fit, metric))]
span = max(np.ptp([r['chi2'] for r in chi2_rows if r['family'] == family]) for family in rows_of) + 4.0

fig, axes = plt.subplots(1, 2, figsize=(7.0, 2.3), sharey=True)
for ax, family in zip(axes, rows_of):
    values = [r['chi2'] for r in chi2_rows if r['family'] == family]
    centre = 0.5 * (min(values) + max(values))
    for row, fit in enumerate(rows_of[family]):
        y = len(rows_of[family]) - 1 - row
        for metric, style in METRICS.items():
            chi2 = value(family, fit, metric)
            if np.isfinite(chi2):
                ax.plot(chi2, y, style['marker'], color=style['color'], ms=7, mec='white', mew=0.9, zorder=3,
                        label=metric if row == 0 else None)
    ax.set_xlim(centre - span / 2, centre + span / 2)
    ax.set_ylim(-0.6, len(rows_of[family]) - 0.4)
    ax.set_yticks(range(len(rows_of[family])))
    ax.set_yticklabels([fit_label[fit] for fit in rows_of[family]][::-1], fontsize=8)
    ax.tick_params(axis='y', length=0, right=False)
    ax.tick_params(axis='x', which='both', labelsize=8, top=False)
    ax.xaxis.set_major_locator(mpl.ticker.MultipleLocator(5))
    ax.xaxis.set_minor_locator(mpl.ticker.MultipleLocator(1))
    ax.grid(axis='y', color='0.85', lw=0.6, zorder=0)
    ax.set_axisbelow(True)
    ax.set_title(FAMILY_LABEL[family], fontsize=9)
fig.supxlabel(r'best-fit $\chi^2$ (72 bins), iterated frozen covariance', fontsize=9, y=0.01)
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc='upper center', ncol=4, frameon=False, fontsize=8, bbox_to_anchor=(0.5, 1.0))
fig.tight_layout(rect=(0, 0.04, 1, 0.92))
fig.subplots_adjust(bottom=0.24)
fig.savefig(FIG_DIR / 'metric_chi2.pdf'); fig.savefig(FIG_DIR / 'metric_chi2.png', dpi=200); plt.show()

CHI2_BY_METRIC = pd.DataFrame(chi2_rows).pivot(index=['family', 'fit'], columns='metric', values='chi2').reindex(columns=list(METRICS))
display(CHI2_BY_METRIC.style.format('{:.1f}', na_rep=''))
pd.DataFrame(chi2_rows).to_csv(TABLE_DIR / 'chi2_by_metric.csv', index=False, float_format='%.6g')